# BadBlueprint full scoring (Colab)
This notebook runs the BadBlueprint full scoring flow using the open-weight model and the vendored harness.

## A. Runtime / GPU check

In [ ]:
import os
import platform
import shutil
import subprocess


def run(cmd):
    print(f"$ {cmd}")
    subprocess.run(cmd, shell=True, check=False)

print("Python:", platform.python_version())
run("nvidia-smi || true")
run("python - <<'PY'
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
PY")
run("free -h")
run("df -h /")

if shutil.which("nvidia-smi") is None:
    print("WARNING: No GPU detected. Full scoring may be extremely slow.")

## B. Environment setup

In [ ]:
import subprocess


def sh(cmd: str):
    print(f"$ {cmd}")
    subprocess.run(cmd, shell=True, check=True)


# Uninstall potentially conflicting packages (safe no-op if missing)
for pkg in ["torchvision", "torchaudio"]:
    sh(f"pip uninstall -y {pkg} || true")

# Install Transformers with serving extras (recommended by HF docs)
# Avoid installing from git main for reproducibility.
sh('pip install -U --quiet "transformers[serving]"')

# Core runtime deps
sh("pip install -U --quiet accelerate safetensors huggingface_hub requests")

# Optional: make downloads more reliable (does nothing if unsupported)
sh("pip install -U --quiet hf_transfer || true")


## C. Model download

In [ ]:
from huggingface_hub import snapshot_download
from pathlib import Path
import os

model_id = "openai/gpt-oss-20b"
local_dir = Path("/content/models/gpt-oss-20b")
local_dir.mkdir(parents=True, exist_ok=True)

hf_token = os.environ.get("HF_TOKEN")

snapshot_download(
    repo_id=model_id,
    local_dir=str(local_dir),
    local_dir_use_symlinks=False,
    token=hf_token,
)

file_count = sum(1 for _ in local_dir.rglob("*"))
print(f"Model downloaded to: {local_dir} (files: {file_count})")

## D. Start local OpenAI-compatible endpoint

In [ ]:
import os
import signal
import subprocess
import time
from pathlib import Path

endpoint = "http://127.0.0.1:8000/v1"
server_log = Path("/content/transformers_server.log")
server_pid = Path("/content/transformers_server.pid")

# Use a stable model identifier (HF serving docs recommend --force-model)
MODEL_ID = "openai/gpt-oss-20b"

def is_pid_running(p: int) -> bool:
    try:
        os.kill(p, 0)
        return True
    except ProcessLookupError:
        return False

def stop_server():
    if not server_pid.exists():
        return
    pid = int(server_pid.read_text().strip())
    try:
        os.killpg(pid, signal.SIGTERM)
        time.sleep(2)
        # Best-effort hard kill if still around
        if is_pid_running(pid):
            os.killpg(pid, signal.SIGKILL)
        print(f"Stopped server process group {pid}")
    except ProcessLookupError:
        print("Server process not running.")
    finally:
        server_pid.unlink(missing_ok=True)

def start_server():
    # Always (re)start cleanly if PID file exists but process not alive
    if server_pid.exists():
        pid = int(server_pid.read_text().strip())
        if not is_pid_running(pid):
            print("Found stale PID file; removing.")
            server_pid.unlink(missing_ok=True)

    if server_pid.exists():
        print("Server appears to be running already. Skipping start.")
        return

    cmd = [
        "transformers",
        "serve",
        "--port",
        "8000",
        "--force-model",
        MODEL_ID,
    ]
    print("Starting server:", " ".join(cmd))
    with server_log.open("w") as log_f:
        proc = subprocess.Popen(
            cmd,
            stdout=log_f,
            stderr=subprocess.STDOUT,
            preexec_fn=os.setsid,  # make process group so we can killpg later
        )
    server_pid.write_text(str(proc.pid))
    time.sleep(5)
    print(f"Server PID: {proc.pid}")
    print(f"Server log: {server_log}")

start_server()
print("Model endpoint:", endpoint)


### Healthcheck

In [ ]:
import time
import requests

endpoint = "http://127.0.0.1:8000/v1"

ok = False
last_err = None
for _ in range(20):
    try:
        resp = requests.get(f"{endpoint}/models", timeout=5)
        if resp.status_code == 200:
            print("Server is healthy.")
            print(resp.json())
            ok = True
            break
        else:
            last_err = f"HTTP {resp.status_code}: {resp.text[:200]}"
    except Exception as exc:
        last_err = str(exc)
    time.sleep(3)

if not ok:
    raise RuntimeError(f"Server healthcheck failed: {last_err}")

## E. Clone repo and prepare submission bundle

In [ ]:
import os
import subprocess
from pathlib import Path


def run(cmd, cwd=None):
    print(f"$ {cmd}")
    subprocess.run(cmd, shell=True, check=True, cwd=cwd)

repo_dir = Path.cwd()
if not (repo_dir / "scripts" / "export_badblueprint_submission.py").exists():
    repo_dir = Path("/content/purple-vanguard-scenarios")
    if not repo_dir.exists():
        run("git clone https://github.com/Purple-Vanguard/purple-vanguard-scenarios.git", cwd=Path("/content"))

run("python scripts/export_badblueprint_submission.py", cwd=repo_dir)
run("python scripts/validate_submission_bundle.py submissions/purple_vanguard/badblueprint", cwd=repo_dir)

## F. Install vendored harness

In [ ]:
import os
import subprocess
import sys
from pathlib import Path
import re
import tomllib

repo_dir = Path.cwd()
if not (repo_dir / "vendor" / "agentbeats-lambda").exists():
    repo_dir = Path("/content/purple-vanguard-scenarios")

subprocess.run("pip install -e vendor/agentbeats-lambda", shell=True, check=True, cwd=repo_dir)

pyproject = repo_dir / "vendor" / "agentbeats-lambda" / "pyproject.toml"
cli_name = None
if pyproject.exists():
    data = tomllib.loads(pyproject.read_text())
    scripts = data.get("project", {}).get("scripts", {})
    if scripts:
        cli_name = sorted(scripts.keys())[0]

if cli_name is None:
    # Fallback: list bin scripts
    bin_dir = Path(sys.executable).parent
    candidates = [p.name for p in bin_dir.iterdir() if p.is_file() and "agent" in p.name]
    cli_name = candidates[0] if candidates else None

if not cli_name:
    raise RuntimeError("Could not detect harness CLI entrypoint.")

print("Detected harness CLI:", cli_name)

## G. Configure harness to use local endpoint

In [ ]:
import os
import re
from pathlib import Path

repo_dir = Path.cwd()
if not (repo_dir / "vendor" / "agentbeats-lambda").exists():
    repo_dir = Path("/content/purple-vanguard-scenarios")

vendor_root = repo_dir / "vendor" / "agentbeats-lambda"
endpoint = "http://127.0.0.1:8000/v1"

# Scan vendor source for explicit env var references (no rg dependency)
ENV_PATTERNS = [
    re.compile(r'os\.environ\[\s*"([A-Z0-9_]+)"\s*\]'),
    re.compile(r"os\.environ\[\s*'([A-Z0-9_]+)'\s*\]"),
    re.compile(r'getenv\(\s*"([A-Z0-9_]+)"\s*\)'),
    re.compile(r"getenv\(\s*'([A-Z0-9_]+)'\s*\)"),
]

def iter_text_files(root: Path):
    for p in root.rglob("*"):
        if p.is_file() and p.suffix in {".py", ".toml", ".yaml", ".yml", ".md"}:
            yield p

env_names = set()
for p in iter_text_files(vendor_root):
    try:
        txt = p.read_text(errors="ignore")
    except Exception:
        continue
    for pat in ENV_PATTERNS:
        for m in pat.findall(txt):
            env_names.add(m)

if not env_names:
    raise RuntimeError("No environment variables found in vendor source. Cannot configure endpoint reliably.")

# Narrow to likely endpoint vars
endpoint_vars = [
    n for n in env_names
    if any(k in n for k in ["BASE_URL", "API_BASE", "ENDPOINT", "HOST", "URL"])
    and "MODEL" not in n
    and "KEY" not in n
]

# Likely api key vars
key_vars = [n for n in env_names if "API_KEY" in n or n.endswith("_KEY")]

# Likely model vars (best-effort)
model_vars = [
    n for n in env_names
    if "MODEL" in n and not any(k in n for k in ["ENDPOINT", "BASE", "URL", "HOST"])
]

# Apply
for n in endpoint_vars:
    os.environ[n] = endpoint

for n in key_vars:
    os.environ.setdefault(n, "DUMMY_KEY")  # do not print, do not log real secrets

for n in model_vars:
    os.environ.setdefault(n, "gpt-oss-20b")

# Also keep a common default for our notebook logic
os.environ.setdefault("MODEL_NAME", "gpt-oss-20b")

print("Configured endpoint:", endpoint)
print("Set endpoint vars:", ", ".join(sorted(endpoint_vars)) if endpoint_vars else "(none found)")
print("Set key vars:", ", ".join(sorted(key_vars)) if key_vars else "(none found)")
print("Set model vars:", ", ".join(sorted(model_vars)) if model_vars else "(none found)")
print("MODEL_NAME:", os.environ.get("MODEL_NAME"))


## H. Run FULL SCORING

In [ ]:
import os
import re
import subprocess
from pathlib import Path

repo_dir = Path.cwd()
if not (repo_dir / "vendor" / "agentbeats-lambda").exists():
    repo_dir = Path("/content/purple-vanguard-scenarios")

results_dir = repo_dir / "results" / "badblueprint"
results_dir.mkdir(parents=True, exist_ok=True)
log_path = results_dir / "full_score.log"

toml_path = repo_dir / "submissions" / "purple_vanguard" / "badblueprint" / "scenario_badblueprint.toml"
if not toml_path.exists():
    raise RuntimeError(f"Scenario TOML not found: {toml_path}")

# Detect CLI from vendor pyproject
pyproject = repo_dir / "vendor" / "agentbeats-lambda" / "pyproject.toml"
cli_name = None
if pyproject.exists():
    import tomllib
    data = tomllib.loads(pyproject.read_text())
    scripts = data.get("project", {}).get("scripts", {})
    if scripts:
        cli_name = sorted(scripts.keys())[0]
if not cli_name:
    raise RuntimeError("Could not detect harness CLI entrypoint.")

# Decide how to run: prefer "<cli> <scenario.toml>" style; fallback to "score" if present
help_txt = ""
try:
    p = subprocess.run([cli_name, "--help"], cwd=repo_dir, capture_output=True, text=True, check=False)
    help_txt = (p.stdout or "") + "\n" + (p.stderr or "")
except Exception:
    help_txt = ""

supports_score_subcmd = bool(re.search(r"(?m)^\s*score\b", help_txt)) or (" score " in help_txt)

# Build command
cmd = None
if supports_score_subcmd:
    cmd = [cli_name, "score", "--mode", "full", str(toml_path)]
else:
    # AgentBeats docs pattern: pass scenario.toml directly
    supports_show_logs = "--show-logs" in help_txt
    cmd = [cli_name, str(toml_path)]
    if supports_show_logs:
        cmd.insert(1, "--show-logs")

print("Running:", " ".join(cmd))
with log_path.open("w") as log_f:
    proc = subprocess.run(cmd, cwd=repo_dir, stdout=log_f, stderr=subprocess.STDOUT)

exit_code = proc.returncode
print("Scoring exit code:", exit_code)

# Collect agent cards if available (best-effort)
def copy_if_exists(src: Path, dst: Path):
    if src.exists():
        dst.write_text(src.read_text())

candidates = list(repo_dir.rglob("agent-card*.json"))
for role in ["green", "attacker", "defender"]:
    hit = None
    for p in candidates:
        if role in p.name.lower():
            hit = p
            break
    if hit:
        copy_if_exists(hit, results_dir / f"agent-card-{role}.json")

exit_code


## I. Write score_status.json

In [ ]:
import json
from datetime import datetime, timezone
from pathlib import Path

repo_dir = Path.cwd()
if not (repo_dir / "vendor" / "agentbeats-lambda").exists():
    repo_dir = Path("/content/purple-vanguard-scenarios")

results_dir = repo_dir / "results" / "badblueprint"
results_dir.mkdir(parents=True, exist_ok=True)

pin_path = repo_dir / "vendor" / "agentbeats-lambda" / "COMMIT_PIN.txt"
commit_pin = pin_path.read_text().strip() if pin_path.exists() else "unknown"

log_path = results_dir / "full_score.log"
exit_code = globals().get("exit_code", None)

success = (exit_code == 0)
notes = "ok"

if exit_code is None:
    success = False
    notes = "exit_code not found (did scoring cell run?)"
elif exit_code != 0:
    # Include last ~40 lines as a hint (no secrets should be present)
    try:
        tail = "\n".join(log_path.read_text(errors="ignore").splitlines()[-40:])
        notes = f"scoring failed (exit_code={exit_code}). log_tail:\n{tail}"
    except Exception:
        notes = f"scoring failed (exit_code={exit_code}). could not read log."

status = {
    "mode": "full",
    "ran_at": datetime.now(timezone.utc).isoformat(),
    "submission_path": "submissions/purple_vanguard/badblueprint",
    "harness_commit_pin": commit_pin,
    "model_endpoint": "http://127.0.0.1:8000/v1",
    "success": bool(success),
    "notes": notes,
}

(results_dir / "score_status.json").write_text(json.dumps(status, indent=2, sort_keys=True))
print("Wrote", results_dir / "score_status.json")


## J. Package results for download

In [ ]:
import tarfile
from pathlib import Path

repo_dir = Path.cwd()
if not (repo_dir / "results" / "badblueprint").exists():
    repo_dir = Path("/content/purple-vanguard-scenarios")

results_dir = repo_dir / "results" / "badblueprint"
archive_path = Path("/content/results_badblueprint_colab.tgz")

with tarfile.open(archive_path, "w:gz") as tar:
    tar.add(results_dir, arcname="results/badblueprint")

size = archive_path.stat().st_size
print(f"Archive created: {archive_path} ({size} bytes)")
print("Download it from the Colab file browser (left sidebar).")


## Cleanup (stop server)

In [ ]:
import os
import signal
import time
from pathlib import Path

server_pid = Path("/content/transformers_server.pid")
if server_pid.exists():
    pid = int(server_pid.read_text())
    try:
        os.killpg(pid, signal.SIGTERM)
        time.sleep(2)
        # Best-effort hard kill if still running
        try:
            os.kill(pid, 0)
            os.killpg(pid, signal.SIGKILL)
        except ProcessLookupError:
            pass
        print(f"Stopped server process group {pid}")
    except ProcessLookupError:
        print("Server process not running.")
    server_pid.unlink(missing_ok=True)
else:
    print("No server PID file found.")
